# nb03 — Four scores, η², decision tree (Figure 6 B, C/D, F) and Supplementary Figure S14

**Plan:** `article_1_revision_implementation_plan_261029.md`, WP3 (four scores, decisions D3 / D7),
WP4 (selection rule), WP9 §9.1 (Figure 6: A and E removed; B η² on the reduced panel; C/D four
scores instead of the radar, with the λ = 0 / λ = 1 anchors; F decision tree with readouts) and
§9.2 (S14 = old S18 reworked: marker-correlation threshold, prediction non-inferiority, label
leakage).

**Status of the selection (read before using 6C/D and 6F).** The rule (MD5 `17d4fe01…`, frozen
2026-11-03) is applied by `r04_apply_rule.py`. Panels 6C/D and 6F read
`output/wp4/selection_after_visual_check.tsv`: the front without the vetoed approaches. On the run
date the visual check is closed (gate G2b, 2026-11-07: 22 accepted approaches) and the two
multi-platform leaves are merged into one, "RNA-seq + microarrays" (gate G4.2, 2026-11-07). Panel 6F
shows only approaches Daniil accepted (switch `TREE_STATUSES`).

| Input | Content |
|---|---|
| `output/wp3/eta_squared.tsv` | mean one-way η² of strategy / method / imputation / post-removal, reduced panel and 87-metric check (r45) |
| `output/wp4/selection_after_visual_check.tsv` | front members per scenario after the vetoes, with gates, four scores, `visual_check` |
| `output/wp4/gates.tsv` | 2,234 approaches with L, N_gate, N_raw, gate results (r04) |
| `output/wp6/gradient_scores.parquet` | gradient runs: λ = 0 / λ = 1 anchors and the label-leakage panel |

**Outputs** (`figures_svg/`): `fig6B_eta_squared`, `fig6CD_four_scores`, `fig6F_decision_tree`,
`figS14a_marker_correlation_gate`, `figS14b_prediction_gate`, `figS14c_label_leakage`; tables
`output/wp9/nb03_*.tsv`.

**Run:** `jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.kernel_name=collagen_3_11 nb03_four_scores_and_tree.ipynb` (from `article_1_revision/`).

Updated 2026-11-07 (third review, gate G4): 6F redrawn in the layout of the submitted Figma tree (4 levels, 4 leaves, multi-platform leaves merged); no label refers to the submitted version. Earlier the same day (gate G2b closed): final selection of 22 accepted approaches, leaf assignment, article palettes (`method_pal`, `strat_pal`).


## §0 Configuration

### §0a Imports

In [ ]:
# Imports. revision_lib is used only for plain-word names and paths (plan WP9).
import json
from pathlib import Path

import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

import revision_lib as rl

PN = rl.PUBLICATION_NAMES  # every label comes from here (plan §1.1)

### §0b Paths and switches

| Switch | Default | Meaning |
|---|---|---|
| `FOUR_SCORE_COLUMNS` | L, N, four scaled bars | columns of Figure 6C/D. The bars are min-max scaled on the main-benchmark range (2,234 approaches), so 0 / 1 mean the same in every experiment |
| `ANCHOR_KIND` | `"capped gradient, all arms"` | the λ = 0 (positive control) and λ = 1 (negative control) reference lines = median over every arm of the 33 capped gradient designs at that λ |
| `TREE_STATUSES` | `["accepted"]` | visual-check statuses shown on the decision-tree leaves; add `"no veto given"` / `"not reviewed"` to see the whole front |
| `TREE_MEMBERS_PER_LEAF` | 3 | at most this many approaches per leaf, in front-rank order |
| `L_THRESHOLD`, `N_MARGIN` | 0.7, 0.05 | the two gates of the frozen rule (steps 1 and 2); copied here only to draw the lines |
| `LEAKAGE_PAIRS` | SVA / ComBat with and without the diagnosis covariate | arms of S14c |

In [ ]:
# Paths and switches.
WP3_DIR = rl.OUTPUT_DIR / "wp3"
WP4_DIR = rl.OUTPUT_DIR / "wp4"
WP6_DIR = rl.OUTPUT_DIR / "wp6"
FIGURES_DIR = rl.REVISION_DIR / "figures_svg"
TABLES_DIR = rl.OUTPUT_DIR / "wp9"
FIGURES_DIR.mkdir(exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

FOUR_SCORE_COLUMNS = [
    "L",
    "N",
    "global_batch_scaled",
    "local_batch_scaled",
    "global_biology_scaled",
    "local_biology_scaled",
]
ANCHOR_KIND = "capped gradient, all arms"
TREE_STATUSES = ["accepted"]
TREE_MEMBERS_PER_LEAF = 3
L_THRESHOLD = 0.7  # rule step 1
N_MARGIN = 0.05  # rule step 2: N ≥ N_raw − 0.05
LEAKAGE_PAIRS = [("04_sva", "04_sva_nocov"), ("05_combat", "05_combat_nocov")]

### §0c Palettes, plain-word names, plot settings

In [ ]:
# Article 1 palettes, copied (not imported) from the frozen figure notebook
# figures_for_article/Finally_assembled_figures_for_article.ipynb, cell 4. method_pal is
# that notebook's tab20 + tab20b palette (saturation + 0.15) over the 33 sorted methods,
# written out as hex so every revision figure uses exactly the submitted colours.
method_pal = {
    "01_raw": "#046cb4",
    "02_median_scaling": "#8bb3e8",
    "03_limma": "#ff7800",
    "04_sva": "#ffa852",
    "05_combat": "#14a014",
    "06_combat_seq": "#7cdf69",
    "07_pycombat": "#d60708",
    "08_inmoose_combatseq": "#ff7270",
    "09_ruv": "#864bbd",
    "10_mnn": "#b790d5",
    "11_harmony": "#8c4536",
    "12_scanorama": "#c48477",
    "13_fsmvn": "#e355b8",
    "14_qsmooth": "#f791bd",
    "15_fsqn_py": "#7f6c6c",
    "16_fsqn_r": "#c7a9a9",
    "17_quantile": "#bcbd06",
    "18_rank": "#dbdb6c",
    "19_tdm": "#00bccf",
    "20_shambhala": "#7cd5e5",
    "21_harmonizr": "#272979",
    "22_tmm": "#3a3ca3",
    "23_vst": "#4c50cf",
    "25_angel": "#7b7ede",
    "26_xpn": "#5d7927",
    "27_dwd": "#85a23a",
    "28_npn": "#adcf4c",
    "29_combat_ref": "#c7db7b",
    "31_ruv3prps": "#8c661c",
    "33_amdbnorm": "#bd971d",
    "34_arsyn": "#e7b02f",
    "36_explobatch": "#e7bf71",
    "38_harman": "#842925",
}
# covariate-free variants of the revision: the hue of the parent method, dashed line
method_pal["04_sva_nocov"] = method_pal["04_sva"]
method_pal["05_combat_nocov"] = method_pal["05_combat"]
strat_pal = {
    "S0_no_removal": "#a1c9f4",
    "A_confirmed_bad": "#ffb482",
    "B_extended_bad": "#8de5a1",
    "C_rnaseq_only": "#ff9f9b",
    "D_malignant_only": "#d0bbff",
    "E1_iterative_r1": "#debb9b",
    "E2_iterative_r2": "#fab0e4",
    "E3_iterative_r3": "#cfcfcf",
    "F_microarray_only": "#fffea3",
    "G_affymetrix_only": "#b9f2f0",
    "H_affymetrix_extended": "#bdb76b",
    "I_rare_batches_removed": "#b095c5",
    "J_ff_only": "#7ecbc4",
    "K_ffpe_only": "#f2c27f",
}
imp_pal = {"strict": "#2979ae", "knn": "#982d22", "softimpute": "#713689"}
post_rm_pal = {False: "#CCCCCC", True: "#333333"}

# Gradient arms; the colour of every arm comes from method_pal (gate G4 review, Daniil
# 2026-11-07: the article palettes in every figure).
ARM_ORDER = [
    "01_raw", "03_limma", "04_sva", "04_sva_nocov", "05_combat", "05_combat_nocov",
    "10_mnn", "11_harmony", "13_fsmvn", "16_fsqn_r", "17_quantile", "33_amdbnorm",
]
ARM_COLORS = {arm: method_pal[arm] for arm in ARM_ORDER}
ARM_LINESTYLES = {arm: ("--" if arm.endswith("_nocov") else "-") for arm in ARM_ORDER}


def arm_label(arm: str) -> str:
    # plain-word method name (covariate-free variants come from PUBLICATION_NAMES)
    return PN.get(arm, rl.method_display_name(arm))


In [ ]:
# Plain-word names of the scores, imputations and visual-check statuses.
SCORE_NAMES = {"L": PN["L"], "N": PN["N"]}
for bar in rl.FOUR_SCORE_BARS:
    SCORE_NAMES[f"{bar}_scaled"] = PN[bar]
IMPUTATION_NAMES = {"strict": "no imputation", "knn": "KNN", "softimpute": "softImpute"}
FACTOR_NAMES = {"Strategy": "Sample-removal strategy", "Method": "Harmonization method",
                "Imputation": "Missing-value imputation", "Post-removal": "Post-removal of batches"}
STATUS_STYLE = {  # marker face per visual-check status
    "accepted": {"facecolor": "#1F3A5F", "label": "Accepted at the visual check"},
    "no veto given": {"facecolor": "#8DA9C4", "label": "On the front, no veto given"},
    "not reviewed": {"facecolor": "white", "label": "On the front, visual check pending"},
}
ANCHOR_COLORS = {"lambda0": "#2B6CB0", "lambda1": "#C53030"}  # balanced = cold, confounded = warm
SCENARIO_ORDER = [
    "FF only",
    "FFPE only",
    "RNA-seq only",
    "Multi-platform (RNA-seq + microarrays)",
    "Microarrays only (not a tree leaf)",
    "No batch removal (S0 only)",
]
SCENARIO_NAMES = {  # pools as they will be named in the text (no internal strategy codes)
    "FF only": "FF only",
    "FFPE only": "FFPE only",
    "RNA-seq only": "RNA-seq only",
    "Multi-platform (RNA-seq + microarrays)": "RNA-seq + microarrays",
    "Microarrays only (not a tree leaf)": "Microarrays only (information only)",
    "No batch removal (S0 only)": "S0 · No removal (information only)",
}


def approach_label(row) -> str:
    # "Method, strategy, imputation[, post-removal]" in plain words
    text = f"{row.method_display}, {rl.STRATEGY_DISPLAY_NAMES[row.strat]}, {IMPUTATION_NAMES[row.imp]}"
    return text + (", post-removal" if row.post_rm else "")

In [ ]:
# Plot settings (SVG + PDF + PNG 200 dpi) and helpers. Font sizes, slot_figsize and
# save_figure are redefined in Figma units by the next cell (Daniil 2026-11-09).
GLOBAL_FONT_SIZE = 7
SMALL_FONT_SIZE = 6  # smallest text on any panel (journal minimum)
plt.rcParams.update({
    "font.size": GLOBAL_FONT_SIZE,
    "axes.labelsize": GLOBAL_FONT_SIZE,
    "axes.titlesize": GLOBAL_FONT_SIZE + 1,
    "xtick.labelsize": GLOBAL_FONT_SIZE - 1,
    "ytick.labelsize": GLOBAL_FONT_SIZE - 1,
    "legend.fontsize": GLOBAL_FONT_SIZE - 1,
    "pdf.fonttype": "truetype",
    "svg.fonttype": "none",
    "figure.dpi": 100,
})
sns.set_style("ticks")


# Panel slots on the full figure page (D2, Daniil 2026-11-08). The page is the Figma frame
# "Figure 4" (node 761-28, 753 × 1150 px) = the 180 × 275 mm text area; every panel is drawn
# at its true size on that page, so GLOBAL_FONT_SIZE is the printed size in pt (≥ 6 pt).
# Copied from figures_svg/PLACEMENT.md (column "slot (mm)"); same dict in nb03–nb05.
PAGE_MM = (180, 275)
PANEL_SLOTS_MM = {
    "fig6B_eta_squared": (85, 50),
    "fig6_gradient_panel": (95, 50),
    "fig6CD_four_scores": (180, 110),
    "fig6F_decision_tree": (180, 110),
    "figS14a_marker_correlation_gate": (180, 85),
    "figS14b_prediction_gate": (180, 85),
    "figS14c_label_leakage": (180, 80),
    "figS16a_robustness_tau": (180, 140),
    "figS16b_front_stability": (180, 115),
    "figS17a_gradient_scores": (180, 125),
    "figS17b_gradient_slopes": (180, 55),
    "figS17c_observational_strategies": (180, 45),
    "figS17d_gradient_scatter": (180, 58),
    "figS17e_gradient_rank_trajectories": (180, 62),
    "figS17f_selected_methods_scores": (180, 105),
    "figS18a_v1_rank_on_test": (180, 50),  # half height (Daniil 2026-11-09)
    "figS18b_v1_regret": (180, 90),  # was S18c (S18b merged into S18a, 2026-11-09)
    "figS2_merged_distributions_all_cohorts": (180, 275),
}
MM_PER_INCH = 25.4


def slot_figsize(stem: str) -> tuple[float, float]:
    """Figure size in inches of the page slot of a panel file (PANEL_SLOTS_MM)."""
    width_mm, height_mm = PANEL_SLOTS_MM[stem]
    return width_mm / MM_PER_INCH, height_mm / MM_PER_INCH


def save_figure(fig: plt.Figure, stem: str) -> None:
    # one file per panel: SVG + PDF + PNG (200 dpi)
    for ext in ("svg", "pdf", "png"):
        fig.savefig(FIGURES_DIR / f"{stem}.{ext}", dpi=200, bbox_inches="tight")
    print(f"saved figures_svg/{stem}.svg/.pdf/.png")


def save_table(table: pd.DataFrame, stem: str) -> None:
    # every plotted table is kept next to the figure inputs
    table.to_csv(TABLES_DIR / f"{stem}.tsv", sep="\t", index=False)
    print(f"saved output/wp9/{stem}.tsv ({len(table)} rows)")

In [ ]:
# Figma units (Daniil 2026-11-09, `daniil_review_261109.md`). Figma imports an SVG with
# 1 SVG pt = 4/3 px, and the figure frame (Figma node 1607-116664) is 756 × 1159 px for the
# 180 × 275 mm page. So every panel is drawn at (slot in Figma px) × 0.75 pt, and every text
# is 7.5 pt = 10 px in Figma. Legends and colour bars ("palettes") may stick out of the slot:
# only the plot area (axes with their tick labels and titles) is fitted to it.
import matplotlib.text as mtext
from matplotlib.transforms import Bbox

FIGMA_FRAME_PX = (756, 1159)
FIGMA_PAGE_MM = (180, 275)
FIGMA_PX_PER_PT = 4 / 3
FIGMA_PX_PER_MM = FIGMA_FRAME_PX[0] / FIGMA_PAGE_MM[0]  # 4.2 px per mm of the page
FIGMA_FONT_PX = 10
FIGMA_FONT_PT = FIGMA_FONT_PX / FIGMA_PX_PER_PT  # 7.5 pt
FIGMA_FIT_LOG = rl.OUTPUT_DIR / "wp14" / "figma_fit" / "nb03_four_scores_and_tree.tsv"
FIGMA_FIT_LOG.parent.mkdir(parents=True, exist_ok=True)

GLOBAL_FONT_SIZE = FIGMA_FONT_PT
SMALL_FONT_SIZE = FIGMA_FONT_PT
plt.rcParams.update({key: FIGMA_FONT_PT for key in (
    "font.size", "axes.labelsize", "axes.titlesize", "xtick.labelsize", "ytick.labelsize",
    "legend.fontsize", "legend.title_fontsize", "figure.titlesize")})


def mm_figsize(width_mm: float, height_mm: float) -> tuple[float, float]:
    """Figure size in inches that lands in Figma at the slot size (page mm → Figma px)."""
    to_inches = FIGMA_PX_PER_MM / FIGMA_PX_PER_PT / 72
    return width_mm * to_inches, height_mm * to_inches


def slot_figsize(stem: str) -> tuple[float, float]:
    """Figure size in inches of the page slot of a panel file (PANEL_SLOTS_MM)."""
    return mm_figsize(*PANEL_SLOTS_MM[stem])


def batch_display_name(batch: str) -> str:
    """Plain batch name: underscores → spaces; microarrays drop the last "Unknown" (library
    preparation is never known for a microarray), RNA-seq keeps its library type."""
    parts = str(batch).split("_")
    if parts[0] != "RNASeq" and len(parts) > 2 and parts[-1] == "Unknown":
        parts = parts[:-1]
    return " ".join(parts)


def is_palette_axis(ax: plt.Axes) -> bool:
    """Colour bars and legend-only axes are palettes: Daniil moves them by hand in Figma."""
    if hasattr(ax, "_colorbar") or getattr(ax, "_figma_palette", False):
        return True
    has_data = bool(ax.lines or ax.collections or ax.images or ax.patches or ax.texts)
    return not ax.axison and not has_data


def plot_extent_px(fig: plt.Figure) -> tuple[float, float]:
    """Width and height in Figma px of the plot area (axes, tick labels, titles, texts on
    the axes, suptitle), without legends and colour bars."""
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    boxes = [ax.get_tightbbox(renderer, bbox_extra_artists=list(ax.texts))
             for ax in fig.axes if ax.get_visible() and not is_palette_axis(ax)]
    boxes += [t.get_window_extent(renderer) for t in fig.texts if t.get_text()]
    if getattr(fig, "_suptitle", None) is not None and fig._suptitle.get_text():
        boxes.append(fig._suptitle.get_window_extent(renderer))
    boxes = [b for b in boxes if b is not None and b.width > 0]
    # a legend-only figure has no plot area: measure the whole figure
    union = Bbox.union(boxes) if boxes else fig.bbox
    px_per_display = 72 / fig.dpi * FIGMA_PX_PER_PT
    return union.width * px_per_display, union.height * px_per_display


def set_all_fonts(fig: plt.Figure, size: float) -> None:
    """Every text of the figure at one size (10 px in Figma)."""
    for ax in fig.axes:
        ax.tick_params(labelsize=size)
    for text in fig.findobj(mtext.Text):
        if text.get_text():
            text.set_fontsize(size)


def fit_to_slot(fig: plt.Figure, width_mm: float, height_mm: float | None,
                n_iterations: int = 8) -> None:
    """Resize the figure until its plot area is width_mm wide (and height_mm high; None keeps
    the current aspect ratio of the plot area). Fonts keep their size, so a few redraws are
    needed; fixed-aspect plots (squares) only shrink to fit."""
    target_w = width_mm * FIGMA_PX_PER_MM
    for _ in range(n_iterations):
        plot_w, plot_h = plot_extent_px(fig)
        target_h = plot_h * target_w / plot_w if height_mm is None else height_mm * FIGMA_PX_PER_MM
        if abs(plot_w - target_w) < 1 and abs(plot_h - target_h) < 1:
            break
        fig_w, fig_h = fig.get_size_inches()
        fig.set_size_inches(max(fig_w * target_w / plot_w, 0.5), max(fig_h * target_h / plot_h, 0.5))
    plot_w, plot_h = plot_extent_px(fig)
    target_h = FIGMA_FRAME_PX[1] if height_mm is None else height_mm * FIGMA_PX_PER_MM
    shrink = min(1.0, target_w / plot_w, target_h / plot_h)
    if shrink < 0.995:  # fixed-aspect plots grew in the other direction: scale the whole panel
        fig.set_size_inches(*(fig.get_size_inches() * shrink))


def log_fit(stem: str, fig: plt.Figure, slot_mm: tuple | None) -> None:
    """One row per saved panel: slot and plot area in Figma px (read by r97_figma_check.py)."""
    plot_w, plot_h = plot_extent_px(fig)
    row = pd.DataFrame([{
        "stem": stem,
        "slot_w_px": None if slot_mm is None else round(slot_mm[0] * FIGMA_PX_PER_MM, 1),
        "slot_h_px": None if slot_mm is None or slot_mm[1] is None else round(slot_mm[1] * FIGMA_PX_PER_MM, 1),
        "plot_w_px": round(plot_w, 1), "plot_h_px": round(plot_h, 1),
    }])
    old = pd.read_csv(FIGMA_FIT_LOG, sep="\t") if FIGMA_FIT_LOG.exists() else row.iloc[:0]
    pd.concat([old[old["stem"] != stem], row]).to_csv(FIGMA_FIT_LOG, sep="\t", index=False)


def save_figure(fig: plt.Figure, stem: str, slot_mm: tuple | None = None,
                keep_fonts: bool = False) -> None:
    """Fonts → 10 Figma px (unless keep_fonts: large clustermaps), plot area → slot (default:
    PANEL_SLOTS_MM[stem] when the notebook has it), then SVG + PDF + PNG (200 dpi)."""
    if slot_mm is None and stem in globals().get("PANEL_SLOTS_MM", {}):
        slot_mm = PANEL_SLOTS_MM[stem]
    if not keep_fonts:
        set_all_fonts(fig, FIGMA_FONT_PT)
    if slot_mm is not None:
        fit_to_slot(fig, *slot_mm)
    log_fit(stem, fig, slot_mm)
    for ext in ("svg", "pdf", "png"):
        fig.savefig(FIGURES_DIR / f"{stem}.{ext}", dpi=200, bbox_inches="tight")
    print(f"saved figures_svg/{stem}.svg/.pdf/.png")


## §1 Figure 6B: η² of the four design factors

Mean over metrics of the one-way η² of each factor, on the reduced panel (22 metrics) and, as a
sensitivity check of the panel reduction, on all 87 metrics. (The 87-metric values equal those of
the first article version, 0.363 / 0.256 / 0.016 / 0.002, which checks the formula; the figure
labels the bar by what it is, not by the version — no comparison with the submitted version.)

In [ ]:
# η² bars: factor × metric set.
eta_squared = pd.read_csv(WP3_DIR / "eta_squared.tsv", sep="\t")
eta_squared["factor_name"] = eta_squared["factor"].map(FACTOR_NAMES)
eta_squared["metric_set_name"] = np.where(eta_squared["n_metrics"] == 87, "All 87 metrics",
                                          "Reduced panel (" + eta_squared["n_metrics"].astype(str) + " metrics)")
assert eta_squared["factor_name"].notna().all()
save_table(eta_squared, "nb03_eta_squared")
factor_order = eta_squared.sort_values("eta2_mean", ascending=False)["factor_name"].unique().tolist()
fig, ax = plt.subplots(figsize=slot_figsize("fig6B_eta_squared"))
sns.barplot(data=eta_squared, y="factor_name", x="eta2_mean", hue="metric_set_name", order=factor_order,
            palette={"All 87 metrics": "#BBBBBB", eta_squared["metric_set_name"].iloc[0]: "#1F3A5F"}, ax=ax)
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", fontsize=SMALL_FONT_SIZE, padding=2)
ax.set_xlabel("Mean η² over metrics")
ax.set_ylabel("")
ax.legend(title="", frameon=False, loc="lower right")
sns.despine(ax=ax)
fig.tight_layout()
save_figure(fig, "fig6B_eta_squared")
plt.show()
print(eta_squared.pivot(index="factor_name", columns="metric_set_name", values="eta2_mean").round(4).to_string())

## §2 Figure 6C/D: four scores of the front members, with the gradient anchors

One row per front member (after the vetoes), grouped by scenario; one column per score. Reference
lines: median over all arms of the capped gradient at λ = 0 (blue, every batch balanced) and λ = 1
(red, one diagnosis per batch). They show what a score looks like when diagnosis and batch are not
confounded and when they are fully confounded. The marker face gives the visual-check status.

### §2a Anchors from the gradient

In [ ]:
# Medians over all arms of the capped gradient at λ = 0 and λ = 1.
gradient_scores = pd.read_parquet(WP6_DIR / "gradient_scores.parquet")
capped = gradient_scores[(gradient_scores["status"] == "ok") & (gradient_scores["kind"] == "gradient")
                         & (gradient_scores["cap"] == "150")]
anchors = pd.DataFrame({
    "score": FOUR_SCORE_COLUMNS,
    "lambda0": [capped.loc[capped["lambda"] == 0.0, s].median() for s in FOUR_SCORE_COLUMNS],
    "lambda1": [capped.loc[capped["lambda"] == 1.0, s].median() for s in FOUR_SCORE_COLUMNS],
})
anchors["anchor_kind"] = ANCHOR_KIND
save_table(anchors, "nb03_four_score_anchors")
print(anchors.round(3).to_string(index=False))

### §2b Front members

In [ ]:
# Load the front after the vetoes and order the rows by scenario and front rank.
front = pd.read_csv(WP4_DIR / "selection_after_visual_check.tsv", sep="\t")
front["scenario"] = pd.Categorical(front["scenario"], SCENARIO_ORDER, ordered=True)
front = front.sort_values(["scenario", "front_rank"]).reset_index(drop=True)
front["approach_label"] = [approach_label(row) for row in front.itertuples()]
front["scenario_name"] = front["scenario"].map(SCENARIO_NAMES)
assert (front["visual_check"] == "accepted").all() and len(front) == 22, "final selection: 22 accepted approaches (gate G2b)"
save_table(front[["scenario_name", "front_rank", "approach_label", "visual_check", *FOUR_SCORE_COLUMNS, "N_gate", "N_raw", "gate_N_classes"]], "nb03_front_four_scores")
print(front.groupby(["scenario", "visual_check"], observed=True).size().unstack(fill_value=0).to_string())

In [ ]:
# Dot matrix: rows = the 22 selected approaches, columns = scores, colour = method (method_pal), anchors as vertical lines.
# Prediction column (D4, Daniil 2026-11-08): the value the gate used — DLBCL vs FL for the
# malignant-only strategy D, 3 classes otherwise (`N_gate`) — with an open grey marker at the
# unharmonized value of the same strategy × imputation (`N_raw`), joined by a thin line.
import textwrap  # column titles wrapped to fit the narrow panels
import matplotlib.ticker as mticker

PLOT_VALUE_COLUMN = {score: score for score in FOUR_SCORE_COLUMNS} | {"N": "N_gate"}
COLUMN_TITLES = SCORE_NAMES | {
    "N": "Cross-batch prediction"  # classes in the legend (10 px titles, 2026-11-09)
}
ANCHORS_COINCIDE = 0.01  # |median λ = 0 − median λ = 1| below this → both lines drawn + a note

n_rows = len(front)
fig, axes = plt.subplots(1, len(FOUR_SCORE_COLUMNS), figsize=slot_figsize("fig6CD_four_scores"), sharey=True)
y_positions = np.arange(n_rows)
scenario_breaks = front.groupby("scenario", observed=True).size().cumsum().to_numpy()[:-1] - 0.5
for ax, score in zip(axes, FOUR_SCORE_COLUMNS):
    values = front[PLOT_VALUE_COLUMN[score]]
    if score == "N":
        ax.hlines(front.index, front["N_raw"], values, color="#999999", linewidth=0.7, zorder=2)
        ax.scatter(front["N_raw"], front.index, s=16, facecolor="white", edgecolor="#777777", linewidth=0.7, zorder=3)
    ax.scatter(values, front.index, s=18, facecolor=front["method"].map(method_pal), edgecolor="#333333", linewidth=0.5, zorder=4)
    anchor = anchors.set_index("score").loc[score]
    # λ = 0 wide dashed under, λ = 1 narrow dotted over: coinciding lines still show both colours
    ax.axvline(anchor["lambda0"], color=ANCHOR_COLORS["lambda0"], linestyle="--", linewidth=1.6, zorder=1)
    ax.axvline(anchor["lambda1"], color=ANCHOR_COLORS["lambda1"], linestyle=":", linewidth=1.0, zorder=2)
    if abs(anchor["lambda0"] - anchor["lambda1"]) < ANCHORS_COINCIDE:  # e.g. local batch: both 0.029
        ax.text(0.5, -0.07, "λ = 0 and λ = 1\nmedians coincide", transform=ax.transAxes, ha="center", va="top",
                fontsize=SMALL_FONT_SIZE, color="#555555")
    if score == "L":
        ax.axvline(L_THRESHOLD, color="#555555", linestyle=":", linewidth=0.9)
    for y in scenario_breaks:
        ax.axhline(y, color="#DDDDDD", linewidth=0.8)
    ax.set_title(textwrap.fill(COLUMN_TITLES[score], 10, break_long_words=False), fontsize=SMALL_FONT_SIZE)
    ax.grid(axis="x", color="#EEEEEE", linewidth=0.6)
    ax.tick_params(axis="x", labelsize=SMALL_FONT_SIZE)
    ax.xaxis.set_major_locator(plt.MaxNLocator(2))  # 2 ticks: 10 px labels on ~20 mm axes
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f"))
    sns.despine(ax=ax)
axes[0].set_yticks(y_positions)
axes[0].set_yticklabels(front["approach_label"], fontsize=SMALL_FONT_SIZE)
axes[0].invert_yaxis()
for start, (scenario, size) in zip(np.r_[0, scenario_breaks + 0.5], front.groupby("scenario", observed=True).size().items()):
    axes[-1].annotate(textwrap.fill(SCENARIO_NAMES[scenario], 16), xy=(1.06, start + size / 2 - 0.5),
                      xycoords=("axes fraction", "data"), va="center", fontsize=SMALL_FONT_SIZE)
handles = [mlines.Line2D([], [], marker="o", linestyle="none", markerfacecolor=method_pal[m], markeredgecolor="#333333",
                         label=rl.method_display_name(m)) for m in sorted(front["method"].unique())]
handles += [
    mlines.Line2D([], [], marker="o", linestyle="-", color="#999999", markerfacecolor="white", markeredgecolor="#777777",
                  linewidth=0.7, label="Prediction of the unharmonized data (same strategy and imputation)"),
    mlines.Line2D([], [], linestyle="none", label="Prediction classes: DLBCL vs FL for D, 3 classes otherwise"),
    mlines.Line2D([], [], color=ANCHOR_COLORS["lambda0"], linestyle="--", linewidth=1.6,
                  label="Gradient, balanced batches (λ = 0), median over all methods"),
    mlines.Line2D([], [], color=ANCHOR_COLORS["lambda1"], linestyle=":", linewidth=1.0,
                  label="Gradient, one diagnosis per batch (λ = 1), median (prediction: 3 classes)"),
    mlines.Line2D([], [], color="#555555", linestyle=":", label=PN["gate_L"]),
]
fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.45, 0.0), ncol=3, frameon=False, fontsize=SMALL_FONT_SIZE)
fig.tight_layout(w_pad=1.6)
save_figure(fig, "fig6CD_four_scores")
plt.show()


## §3 Figure 6F: decision tree

Layout of the submitted tree, which Daniil drew in Figma (`figures_for_article/current_figures_tables_for_article_260915/Figure 6.jpg`,
panel F): step labels on the left, then (1) biomaterial / platform type, (2) recommended method,
(3) recommended imputation, (4) the checks on the user's own data and the optional post-removal.
Daniil finishes the panel by hand in Figma (gate G4.2); this is the data-driven draft.

Content: the approaches accepted at the visual check (`TREE_STATUSES`) per leaf, read from
`selection_after_visual_check.tsv` (+ `leaf_assignment.tsv`). The two multi-platform leaves are
merged into "RNA-seq + microarrays" (G4.2). Methods are listed in front-rank order; the first is
the recommendation, the others the alternatives. Imputation: of the recommended (first) method —
"Any" when its accepted approaches cover all three settings, otherwise the settings that occur. The checks box carries the
pass values of the frozen rule (marker correlation ≥ 0.70; cross-batch prediction not below the
unharmonized data by more than 0.05) and the visual inspection.


In [ ]:
# Leaf table: accepted approaches per tree leaf (tree_leaf_name from r04 + leaf_assignment.tsv).
TREE_LEAVES = ["FF only", "FFPE only", "RNA-seq only", "RNA-seq + microarrays"]
LEAF_TITLES = {  # box text of level 1, as in the submitted Figma tree
    "FF only": "FF only\nRNA-seq + microarrays",
    "FFPE only": "FFPE only\nRNA-seq + microarrays",
    "RNA-seq only": "RNA-seq only\n(any biomaterial)",
    "RNA-seq + microarrays": "RNA-seq + microarrays\n(any biomaterial)",
}
LEAF_COLORS = {  # level-1 fill: strategy colour (strat_pal); the merged leaf the D colour
    "FF only": strat_pal["J_ff_only"],
    "FFPE only": strat_pal["K_ffpe_only"],
    "RNA-seq only": strat_pal["C_rnaseq_only"],
    "RNA-seq + microarrays": strat_pal["D_malignant_only"],
}
IMPUTATION_BOX_COLORS = {"Any": "#1F4E79", "knn": imp_pal["knn"], "softimpute": imp_pal["softimpute"], "strict": imp_pal["strict"]}
tree_members = front[front["tree_leaf_name"].isin(TREE_LEAVES) & front["visual_check"].isin(TREE_STATUSES)].copy()
leaf_members = tree_members.groupby("tree_leaf_name", observed=True).head(TREE_MEMBERS_PER_LEAF)
leaf_table = tree_members[["tree_leaf_name", "front_rank", "approach_label", "method", "strat", "imp", "post_rm", "L",
                           "N_gate", "N_raw", "gate_N_classes", "local_batch", "local_biology"]].copy()
save_table(leaf_table, "nb03_decision_tree_leaves")
print(leaf_table.drop(columns=["method", "strat"]).round(3).to_string(index=False))
assert set(leaf_table["tree_leaf_name"]) == set(TREE_LEAVES), set(TREE_LEAVES) - set(leaf_table["tree_leaf_name"])
assert leaf_table["L"].ge(L_THRESHOLD).all() and (leaf_table["N_gate"] >= leaf_table["N_raw"] - N_MARGIN).all()


def leaf_summary(leaf: str) -> dict:
    # methods in front-rank order (first = recommendation), their strategy notes, imputations, post-removal
    rows = tree_members[tree_members["tree_leaf_name"] == leaf].sort_values("front_rank")
    methods = list(dict.fromkeys(rows["method"]))
    notes = {}
    for method in methods:
        strategies = list(dict.fromkeys(rows.loc[rows["method"] == method, "strat"]))
        # leaf names are plain ("FF only"); the note shows the strategy as "code · name"
        own = [rl.STRATEGY_DISPLAY_NAMES[s] for s in strategies if rl.STRATEGY_PLAIN_NAMES[s] != leaf]
        notes[method] = ", ".join(own)
    imputations = set(rows.loc[rows["method"] == methods[0], "imp"])  # of the recommended method
    imputation = "Any" if imputations == {"strict", "knn", "softimpute"} else " / ".join(
        IMPUTATION_NAMES[i] for i in ("softimpute", "knn", "strict") if i in imputations)
    return {"methods": methods, "notes": notes, "imputation": imputation,
            "imputation_key": "Any" if imputation == "Any" else ("knn" if "knn" in imputations else sorted(imputations)[0]),
            "post_removal": sorted(set(rows["post_rm"]))}


tree_summary = {leaf: leaf_summary(leaf) for leaf in TREE_LEAVES}
for leaf, summary in tree_summary.items():
    print(leaf, "|", [rl.method_display_name(m) for m in summary["methods"]], "|", summary["imputation"],
          "| post-removal", summary["post_removal"])


In [ ]:
# Draw the tree in the layout of the submitted Figma panel (levels 1–4, step labels on the left).
import textwrap

from matplotlib.colors import to_rgb
from matplotlib.patches import FancyBboxPatch


def lighten(color: str, amount: float) -> tuple:
    # mix a colour with white (0 = unchanged, 1 = white)
    r, g, b = to_rgb(color)
    return (r + (1 - r) * amount, g + (1 - g) * amount, b + (1 - b) * amount)


def darken(color: str, amount: float) -> tuple:
    r, g, b = to_rgb(color)
    return (r * (1 - amount), g * (1 - amount), b * (1 - amount))


def box(ax, x, y, w, h, text, face, edge, text_color="#1A1A1A", weight="bold", size=SMALL_FONT_SIZE, style="normal", lw=1.6):
    ax.add_patch(FancyBboxPatch((x - w / 2, y - h / 2), w, h, boxstyle="round,pad=0.004,rounding_size=0.012",
                                facecolor=face, edgecolor=edge, linewidth=lw, zorder=3))
    ax.text(x, y, text, ha="center", va="center", fontsize=size, fontweight=weight, fontstyle=style,
            color=text_color, zorder=4, linespacing=1.25)


def link(ax, x1, y1, x2, y2):
    ax.plot([x1, x2], [y1, y2], color="#A6A6A6", linewidth=1.4, zorder=1)


def method_box_text(summary: dict) -> str:
    first, *others = summary["methods"]
    lines = [rl.method_display_name(first) + (f"\n({textwrap.fill(summary['notes'][first], 24)})" if summary["notes"][first] else "")]
    if others:
        names = [rl.method_display_name(m) for m in others]
        lines.append("alt: " + ",\n".join(", ".join(names[i:i + 2]) for i in range(0, len(names), 2)))
    return "\n".join(lines)


fig, ax = plt.subplots(figsize=slot_figsize("fig6F_decision_tree"))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_axis_off()
LEVEL_Y = {"root": 0.94, 1: 0.76, 2: 0.56, 3: 0.39, 4: 0.20, 5: 0.06}
STEP_LABELS = {1: "Biomaterial /\nplatform type", 2: "Recommended\nmethod", 3: "Recommended\nimputation",
               4: "Check on your\nown data", 5: "QC and\npost-removal"}
for step, label in STEP_LABELS.items():  # left column: numbered step labels
    y = LEVEL_Y[step]
    ax.add_patch(plt.Circle((0.025, y), 0.016, facecolor="white", edgecolor="#1A1A1A", linewidth=1.3, transform=ax.transData, zorder=3))
    ax.text(0.025, y, str(step), ha="center", va="center", fontsize=GLOBAL_FONT_SIZE, fontweight="bold", zorder=4)
    box(ax, 0.105, y, 0.12, 0.085, label, "#F2F2F2", "#D0D0D0", text_color="#555555", weight="normal", size=SMALL_FONT_SIZE, lw=1.0)

leaf_x = np.linspace(0.28, 0.895, len(TREE_LEAVES))
leaf_w = 0.192
box(ax, 0.59, LEVEL_Y["root"], 0.24, 0.07, "YOUR DATASET\n(bulk transcriptomics)", "#E8EAF6", "#283593",
    text_color="#283593", style="italic", lw=2)
for x, leaf in zip(leaf_x, TREE_LEAVES):
    summary = tree_summary[leaf]
    leaf_color = LEAF_COLORS[leaf]
    first_method_color = method_pal[summary["methods"][0]]
    link(ax, 0.59, LEVEL_Y["root"] - 0.035, x, LEVEL_Y[1] + 0.045)
    box(ax, x, LEVEL_Y[1], leaf_w, 0.09, LEAF_TITLES[leaf], lighten(leaf_color, 0.25), darken(leaf_color, 0.45),
        text_color=darken(leaf_color, 0.65))
    link(ax, x, LEVEL_Y[1] - 0.045, x, LEVEL_Y[2] + 0.065)
    box(ax, x, LEVEL_Y[2], leaf_w, 0.13, method_box_text(summary), lighten(first_method_color, 0.55),
        darken(first_method_color, 0.25), weight="normal", style="italic", size=SMALL_FONT_SIZE)
    link(ax, x, LEVEL_Y[2] - 0.065, x, LEVEL_Y[3] + 0.03)
    box(ax, x, LEVEL_Y[3], leaf_w * 0.92, 0.06, summary["imputation"], IMPUTATION_BOX_COLORS[summary["imputation_key"]],
        "#111111", text_color="white", lw=1.6)
    link(ax, x, LEVEL_Y[3] - 0.03, 0.59, LEVEL_Y[4] + 0.07)
checks = (f"Marker correlation with the unharmonized data ≥ {L_THRESHOLD:.2f}\n"
          f"Cross-batch diagnosis prediction ≥ unharmonized − {N_MARGIN:.2f}, above its permutation null\n"
          "Visual inspection: batches of the same diagnosis mix, diagnoses stay apart")
box(ax, 0.59, LEVEL_Y[4], 0.7, 0.14, checks, "#F5F5F5", "#9E9E9E", weight="normal", size=SMALL_FONT_SIZE)
link(ax, 0.59, LEVEL_Y[4] - 0.07, 0.59, LEVEL_Y[5] + 0.03)
box(ax, 0.59, LEVEL_Y[5], 0.34, 0.07, "Optional: post-removal of batches\nthat stay apart (check the samples lost)",
    "#FCE8E6", "#C62828", weight="normal", size=SMALL_FONT_SIZE, lw=1.6)
save_figure(fig, "fig6F_decision_tree")
plt.show()


## §4 Supplementary Figure S14a: marker-correlation threshold

L (mean per-gene Spearman ρ of the 56 marker genes before vs after harmonization) of the 2,234
analysed approaches by method, with the 0.7 gate. Methods ordered by median.

In [ ]:
# Load the gate table and draw L per method.
gates = pd.read_csv(WP4_DIR / "gates.tsv", sep="\t")
gates = gates[gates["in_analysis_set"]]
method_order = gates.groupby("method_display")["L"].median().sort_values().index.tolist()
display_pal = {display: method_pal[code] for display, code in gates.groupby("method_display")["method"].first().items()}  # method_pal by plain name
fig, ax = plt.subplots(figsize=slot_figsize("figS14a_marker_correlation_gate"))
sns.boxplot(data=gates, x="method_display", y="L", order=method_order, hue="method_display", hue_order=method_order,
            palette=display_pal, legend=False, fliersize=0, width=0.6, boxprops={"alpha": 0.75}, ax=ax)
sns.stripplot(data=gates, x="method_display", y="L", order=method_order, color="#333333", size=1.6, alpha=0.5, ax=ax)
ax.axhline(L_THRESHOLD, color="#C53030", linestyle="--", linewidth=1)
ax.set_xlabel("")
ax.set_ylabel(PN["L"])
ax.tick_params(axis="x", rotation=60)
for label in ax.get_xticklabels():
    label.set_horizontalalignment("right")
sns.despine(ax=ax)
fig.tight_layout()
save_figure(fig, "figS14a_marker_correlation_gate")
plt.show()
l_pass = gates.groupby("method_display")["gate_L_pass"].mean().sort_values()
save_table(l_pass.rename("share_passing").reset_index(), "nb03_gate_L_pass_by_method")
print(f"approaches passing gate L: {int(gates['gate_L_pass'].sum())} of {len(gates)}")
print(l_pass.round(2).head(8).to_string())

## §5 Supplementary Figure S14b: prediction non-inferiority

Difference between the cross-batch prediction F1 of each approach and that of the unharmonized
data of the same strategy, imputation and post-removal setting (2-class F1 for malignant samples
only, 3-class otherwise). The gate passes at ≥ −0.05 and when the F1 is above its permutation null.

In [ ]:
# N − N_raw per method; colour = above the permutation null or not.
gates["N_difference"] = gates["N_gate"] - gates["N_raw"]
gates["above_null"] = np.where(gates["gate_N_above_null"], "Above the permutation null (p < 0.05)", "Not above the permutation null")
fig, ax = plt.subplots(figsize=slot_figsize("figS14b_prediction_gate"))
sns.boxplot(data=gates, x="method_display", y="N_difference", order=method_order, hue="method_display", hue_order=method_order,
            palette=display_pal, legend=False, fliersize=0, width=0.6, boxprops={"alpha": 0.55}, ax=ax)
sns.stripplot(data=gates, x="method_display", y="N_difference", order=method_order, hue="above_null", size=1.8, alpha=0.6,
              palette={"Above the permutation null (p < 0.05)": "#1F3A5F", "Not above the permutation null": "#C53030"}, ax=ax)
ax.axhline(-N_MARGIN, color="#C53030", linestyle="--", linewidth=1)
ax.axhline(0, color="#999999", linewidth=0.7)
ax.set_xlabel("")
ax.set_ylabel("Prediction F1 minus F1 of the\nunharmonized data")
ax.legend(title="", frameon=False, loc="lower left", markerscale=3)
ax.tick_params(axis="x", rotation=60)
for label in ax.get_xticklabels():
    label.set_horizontalalignment("right")
sns.despine(ax=ax)
fig.tight_layout()
save_figure(fig, "figS14b_prediction_gate")
plt.show()
print(f"approaches passing gate N: {int(gates['gate_N_pass'].sum())} of {len(gates)}; "
      f"non-inferior {int(gates['gate_N_noninferior'].sum())}, above null {int(gates['gate_N_above_null'].sum())}")

## §6 Supplementary Figure S14c: label leakage (covariate vs no covariate)

SVA and ComBat use the diagnosis as a covariate; their covariate-free variants do not. If passing
the diagnosis leaks the label into the corrected data, the covariate arms should keep diagnosis
separation even when batch and diagnosis coincide (high V), and their cross-batch prediction should
stay above the covariate-free arm. Mean ± SD over the 3 replicates of the capped gradient.

In [ ]:
# Leakage panel: N and the two diagnosis bars vs V for the covariate pairs.
leakage_scores = ["N", "global_biology_scaled", "local_biology_scaled"]
leakage_runs = capped[capped["arm"].isin([arm for pair in LEAKAGE_PAIRS for arm in pair])]
leakage_summary = (
    leakage_runs.groupby(["arm", "lambda"])
    .agg(v_mean=("cramers_v_major_group", "mean"), **{f"{s}_mean": (s, "mean") for s in leakage_scores},
         **{f"{s}_sd": (s, "std") for s in leakage_scores})
    .reset_index()
)
save_table(leakage_summary, "nb03_label_leakage")
fig, axes = plt.subplots(1, len(leakage_scores), figsize=slot_figsize("figS14c_label_leakage"), sharex=True)
for ax, score in zip(axes, leakage_scores):
    for arm in [arm for pair in LEAKAGE_PAIRS for arm in pair]:
        rows = leakage_summary[leakage_summary["arm"] == arm].sort_values("v_mean")
        ax.errorbar(rows["v_mean"], rows[f"{score}_mean"], yerr=rows[f"{score}_sd"], color=ARM_COLORS[arm],
                    linestyle=ARM_LINESTYLES[arm], marker="o", markersize=3, capsize=2, linewidth=1.3, label=arm_label(arm))
    ax.set_xlabel(PN["cramers_v"])
    ax.set_ylabel(SCORE_NAMES[score])
    sns.despine(ax=ax)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.0), ncol=4, frameon=False)
fig.tight_layout()
save_figure(fig, "figS14c_label_leakage")
plt.show()
print(leakage_summary[leakage_summary["lambda"].isin([0.0, 1.0])][["arm", "lambda", "N_mean", "local_biology_scaled_mean"]].round(3).to_string(index=False))

## §7 Notes (gate G4 answered 2026-11-07)

- 6F: the two multi-platform leaves are merged (G4.2); Daniil redraws the final tree in Figma, this
  panel is the data-driven draft in the same layout. No label refers to the submitted version.
- 6B: the 87-metric bar is a sensitivity check of the panel reduction, labelled "All 87 metrics".

**Figure review 2026-11-08** (`daniil_review_261108.md`, plan `figure_review_corrections_plan_261108.md`):

- *Why do FSQN R and FSMVN in strategy D show a prediction of about 0.5?* The 6C/D column used
  to show the 3-class score (FSQN R 0.556, FSMVN 0.534). D holds malignant samples only, so its
  gate uses DLBCL vs FL (decision 2026-11-03): FSQN R 0.645, FSMVN 0.651, against 0.622 for the
  unharmonized data of the same strategy and imputation — no loss. The column now shows the gate
  value, with an open grey marker at the unharmonized value (D4). The two methods are on D's front
  because they remove the most batch signal in D; ComBat is the diagnosis end of that front.
- *Why is there no λ = 0 line in the local-batch panel?* It was hidden: the median local batch
  mixing over all gradient arms is 0.029 at λ = 0 and at λ = 1, so the λ = 1 line covered the
  λ = 0 line. Now λ = 0 is wide dashed under a narrow dotted λ = 1 line, and the panel notes that
  the two medians coincide.
- Every panel is drawn at its size on the 180 × 275 mm page (`PANEL_SLOTS_MM`, D2), fonts 6–8 pt.
